In [1]:
# --- الخلية 1: التثبيت الشامل (Brute Force Installation) ---
import sys
import os
import glob
import subprocess

print("⚙️ جاري البحث عن ملفات التثبيت (.whl) وتثبيتها بالجملة...")

# 1. العثور على جميع ملفات whl في كل مجلدات Input
# هذا النمط سيجد الملفات أينما كانت مخبأة
all_whls = glob.glob("/kaggle/input/**/*.whl", recursive=True)

print(f"📦 تم العثور على {len(all_whls)} ملف مكتبة.")

# 2. ترتيب الملفات (نحاول تثبيت الاعتماديات الصغيرة قبل الكبيرة)
# ليس شرطاً أساسياً لكنه أفضل
all_whls = sorted(all_whls, key=lambda x: len(x))

# 3. تثبيت كل شيء!
success_count = 0
for path in all_whls:
    # نتجاوز الملفات التي لا علاقة لها بمشروعنا لتوفير الوقت (اختياري)
    # لكن سنثبت كل شيء للأمان
    try:
        filename = os.path.basename(path)
        # طباعة خفيفة
        # print(f"تركيب: {filename}...")
        
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", 
            path, 
            "--no-deps",     # لا تبحث في الإنترنت
            "--ignore-installed" # ثبت حتى لو كان موجوداً لضمان النسخة
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL) # إخفاء التفاصيل لعدم تشويش الشاشة
        
        success_count += 1
    except Exception as e:
        print(f"⚠️ تجاوز: {filename} (قد تكون غير متوافقة)")

print(f"✅ تمت عملية التثبيت! تم تثبيت {success_count} مكتبة.")

# 4. التأكد من المسارات (خطة الطوارئ)
# إذا كانت هناك مجلدات تحتوي على أكواد بايثون، نضيفها للمسار
src_dirs = glob.glob("/kaggle/input/**/ultralytics", recursive=True)
for d in src_dirs:
    parent = os.path.dirname(d)
    if parent not in sys.path:
        sys.path.append(parent)
        print(f"🔗 تمت إضافة المسار: {parent}")

print("🏁 النظام جاهز 100%. شغل الخلية 2 الآن.")

⚙️ جاري البحث عن ملفات التثبيت (.whl) وتثبيتها بالجملة...
📦 تم العثور على 95 ملف مكتبة.
⚠️ تجاوز: torch-2.4.0-cp310-cp310-manylinux1_x86_64.whl (قد تكون غير متوافقة)
⚠️ تجاوز: pillow-10.4.0-cp310-cp310-manylinux_2_28_x86_64.whl (قد تكون غير متوافقة)
⚠️ تجاوز: torchvision-0.19.0-cp310-cp310-manylinux1_x86_64.whl (قد تكون غير متوافقة)
⚠️ تجاوز: scipy-1.14.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (قد تكون غير متوافقة)
⚠️ تجاوز: numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (قد تكون غير متوافقة)
⚠️ تجاوز: PyYAML-6.0.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (قد تكون غير متوافقة)
⚠️ تجاوز: pandas-2.2.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (قد تكون غير متوافقة)
⚠️ تجاوز: triton-3.0.0-1-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (قد تكون غير متوافقة)
⚠️ تجاوز: contourpy-1.2.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (قد تكون غير متوافقة)
⚠️ تجاوز: kiwisolver-1.4.5-cp310-cp3

In [2]:
# --- الخلية 2: استيراد المكتبات وتجهيز البيئة ---
import numpy as np
import pandas as pd
import cv2
import torch
import os
import glob
from tqdm import tqdm
import matplotlib.pyplot as plt

# مكتبات معالجة الإشارات (Signal Processing)
from scipy.signal import savgol_filter, resample, find_peaks, butter, filtfilt
from skimage.measure import ransac, LineModelND

# محاولة استيراد المكتبات الخارجية التي ثبتناها للتو
try:
    import segmentation_models_pytorch as smp
    from ultralytics import YOLO
    print("✅ نجاح! تم استيراد (SMP) و (YOLO) بنجاح.")
except ImportError as e:
    print(f"❌ خطأ فادح: فشل الاستيراد! تأكد من تشغيل الخلية 1 بنجاح.\nالخطأ: {e}")
    # إيقاف التنفيذ لتنبيهك
    raise e

# إعداد الجهاز (GPU/CPU)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🚀 العمليات ستتم باستخدام: {DEVICE}")

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

✅ نجاح! تم استيراد (SMP) و (YOLO) بنجاح.
🚀 العمليات ستتم باستخدام: cpu


In [3]:
# --- الخلية 3: محرك الرسم الحي (Ultimate Renderer) - [M3: UPDATED] ---
# إعداد قاعدة البيانات (تحميل مرة واحدة فقط)
DB_DIR = "physionet_db"
if not os.path.exists(DB_DIR):
    os.makedirs(DB_DIR)
    print("⬇️ جاري تحميل سجلات PTB-XL الأساسية...")
    try:
        # تحميل عينة كافية لضمان التنوع
        records = wfdb.get_record_list('ptb-xl')
        selected_records = records[:200] 
        wfdb.dl_database('ptb-xl', os.getcwd() + "/" + DB_DIR, selected_records)
        print(f"✅ تم تحميل {len(selected_records)} سجل.")
    except Exception as e:
        print(f"⚠️ تحذير: فشل التحميل ({e})، سيتم استخدام التوليد الاحتياطي.")

class UltimateRenderer:
    def __init__(self, db_dir):
        self.db_dir = db_dir
        self.records = [f.split('.')[0] for f in os.listdir(db_dir) if f.endswith('.dat')] if os.path.exists(db_dir) else []
        
    def get_real_signal(self):
        """سحب إشارة عشوائية من PTB-XL"""
        if not self.records:
            t = np.linspace(0, 4, 2000); return np.sin(2 * np.pi * 5 * t) # Fallback
            
        try:
            rec_name = random.choice(self.records)
            record, meta = wfdb.rdsamp(f"{self.db_dir}/{rec_name}")
            lead_idx = random.randint(0, record.shape[1] - 1)
            signal = record[:, lead_idx]
            signal = np.nan_to_num(signal)
            
            # قص عشوائي (Zoom in/out simulation)
            crop_len = random.randint(1000, 3000)
            if len(signal) > crop_len:
                start = random.randint(0, len(signal) - crop_len)
                return signal[start : start+crop_len]
            return signal
        except:
            return np.random.randn(2000)

    def render_to_memory(self, signal):
        """الرسم بدقة عالية (DPI=200) في الذاكرة مباشرة"""
        # 3. الشبكة المتغيرة (Distractor)
        grid_color = random.choice(['red', '#ffb6c1', '#cfcfcf', '#e0e0e0', 'pink'])
        grid_alpha = random.uniform(0.3, 0.8)
        bg_color = random.choice(['white', '#fffdf5', '#f0f0f0']) 
        
        fig_h, fig_w = 3, 8
        dpi = 200 # حسب الطلب لضمان وضوح الحواف
        
        # --- أ. توليد القناع (Mask Generation) ---
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
        ax.plot(signal, color='white', linewidth=3.0) 
        ax.set_ylim(np.min(signal), np.max(signal))
        ax.axis('off')
        fig.patch.set_facecolor('black')
        
        fig.canvas.draw()
        mask = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        mask = mask.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)
        _, mask = cv2.threshold(mask, 10, 255, cv2.THRESH_BINARY)
        plt.close(fig)

        # --- ب. الرسم الأولي (Rendering) ---
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
        ax.set_facecolor(bg_color)
        
        # رسم الشبكة
        ax.minorticks_on()
        ax.grid(which='major', linestyle='-', linewidth=0.8, color=grid_color, alpha=grid_alpha)
        ax.grid(which='minor', linestyle=':', linewidth=0.4, color=grid_color, alpha=grid_alpha*0.5)
        
        # رسم الإشارة (محاكاة ألوان الحبر المختلفة)
        line_color = random.choice(['black', 'blue', '#000033']) 
        ax.plot(signal, color=line_color, linewidth=random.uniform(1.0, 1.8))
        
        ax.axis('off')
        ax.set_ylim(np.min(signal), np.max(signal))
        fig.patch.set_facecolor(bg_color)
        
        fig.canvas.draw()
        img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        # نحتفظ بها BGR هنا لأن OpenCV يفضل ذلك للمعالجة اللاحقة (Distractors)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR) 
        plt.close(fig)
        
        return img, mask

print("✅ تم تحديث الخلية 3: محرك الرسم الحي جاهز (DPI=200).")

⬇️ جاري تحميل سجلات PTB-XL الأساسية...
⚠️ تحذير: فشل التحميل (name 'wfdb' is not defined)، سيتم استخدام التوليد الاحتياطي.
✅ تم تحديث الخلية 3: محرك الرسم الحي جاهز (DPI=200).


In [4]:
# --- الخلية 22: المحرك البلاتيني (Phase 14+15) - [FINAL FIXED BUILD] ---
import torch
from ultralytics import YOLO
import pandas as pd
import numpy as np
import cv2
import os
import glob
from tqdm import tqdm
from scipy.signal import savgol_filter, resample, find_peaks, butter, filtfilt
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
from skimage.measure import ransac

# --- 1. إعداد المسارات ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TEST_CSV_PATH = "/kaggle/input/physionet-ecg-image-digitization/test.csv"
IMAGE_DIR = "/kaggle/input/physionet-ecg-image-digitization"
LEAD_NAMES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

# --- 2. تحميل النماذج (باستخدام مسارات Input الدائمة) ---
print("⚙️ جاري تحميل المحرك البلاتيني (Phase 14+15)...")

# مسارات النماذج في الداتاسيت المرفوعة
YOLO_MODEL_PATH = "/kaggle/input/ecg-final-models/best.pt"
UNET_MODEL_PATH = "/kaggle/input/ecg-final-models/best_model_phase10.pth"

# 1. تحميل YOLO
if os.path.exists(YOLO_MODEL_PATH):
    yolo_model = YOLO(YOLO_MODEL_PATH)
    print(f"✅ YOLO (Combat): مفعل (تم التحميل من Input).")
else:
    # خطة بديلة: البحث في المكان القديم
    fallback_path = "runs/detect/ecg_combat_detector/weights/best.pt"
    if os.path.exists("best.pt"): fallback_path = "best.pt"
    
    if os.path.exists(fallback_path):
        yolo_model = YOLO(fallback_path)
        print(f"✅ YOLO (Combat): مفعل (تم التحميل من Working).")
    else:
        print("⚠️ تحذير: YOLO غير موجود! تأكد من المسار.")
        yolo_model = None

# 2. تحميل U-Net
unet_model = smp.Unet(encoder_name="resnet50", encoder_weights=None, in_channels=3, classes=1, decoder_attention_type="scse")

if os.path.exists(UNET_MODEL_PATH):
    unet_model.load_state_dict(torch.load(UNET_MODEL_PATH, map_location=DEVICE))
    print("💎 U-Net (Phase 10): مفعل (تم التحميل من Input).")
else:
    # خطة بديلة
    if os.path.exists("best_model_phase10.pth"):
        unet_model.load_state_dict(torch.load("best_model_phase10.pth", map_location=DEVICE))
        print("💎 U-Net (Phase 10): مفعل (تم التحميل من Working).")
    else:
        print(f"⚠️ خطأ فادح: لم يتم العثور على {UNET_MODEL_PATH}!")

unet_model.to(DEVICE)
unet_model.eval()

# --- 3. دوال المعالجة الطبية (Phase 14) ---

def apply_high_pass_filter(data, cutoff=0.5, fs=500, order=5):
    """ إزالة انحراف الخط الأساسي """
    if len(data) < order * 3: return data
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    return filtfilt(b, a, data)

def smart_einthoven_fix(leads):
    """ قانون أينتهوفن الذكي: II = I + III """
    if 'I' in leads and 'II' in leads and 'III' in leads:
        l = min(len(leads['I']), len(leads['II']), len(leads['III']))
        I = leads['I'][:l]; II = leads['II'][:l]; III = leads['III'][:l]
        residual = (I + III) - II
        leads['II'][:l] += residual / 3.0
        leads['I'][:l]  -= residual / 3.0
        leads['III'][:l]-= residual / 3.0
    return leads

# --- 4. دوال التنبؤ المتقدمة (Phase 15: TTA + Helpers) ---

def suppress_vertical_artifacts(prob_map, threshold_factor=2.5):
    col_means = np.mean(prob_map, axis=0)
    global_median = np.median(col_means)
    mask = np.ones_like(col_means)
    artifact_indices = col_means > (global_median * threshold_factor + 0.05)
    mask[artifact_indices] = 0.1
    return prob_map * mask[None, :]

def predict_with_tta(image, model):
    """ TTA: دمج التنبؤ العادي مع التنبؤ المعكوس """
    h, w = image.shape[:2]; target_h = 256; scale = target_h / h; new_w = int(w * scale)
    img_rz = cv2.resize(image, (new_w, target_h))
    
    inp = torch.from_numpy(img_rz).permute(2,0,1).float().unsqueeze(0).to(DEVICE) / 255.0
    img_flip = cv2.flip(img_rz, 1)
    inp_flip = torch.from_numpy(img_flip).permute(2,0,1).float().unsqueeze(0).to(DEVICE) / 255.0
    
    with torch.no_grad():
        prob_orig = torch.sigmoid(model(inp)).cpu().numpy()[0][0]
        prob_flip = torch.sigmoid(model(inp_flip)).cpu().numpy()[0][0]
        
    prob_flip_back = cv2.flip(prob_flip, 1)
    final_prob = (prob_orig + prob_flip_back) / 2.0
    final_prob = suppress_vertical_artifacts(final_prob)
    
    return final_prob, scale

def adaptive_viterbi(prob_map):
    H, W = prob_map.shape
    snr = np.percentile(np.max(prob_map, axis=0), 95) / (np.percentile(np.max(prob_map, axis=0), 10) + 1e-6)
    lam = 0.05 if snr > 5 else (0.5 if snr > 2 else 2.0)
    dp = np.full((H, W), np.inf); par = np.zeros((H, W), int)
    dp[:, 0] = -np.log(prob_map[:, 0] + 1e-6)
    for x in range(1, W):
        prev_col = dp[:, x-1]
        current_probs = -np.log(prob_map[:, x] + 1e-6)
        for y in range(H):
            window_size = 5 if snr > 3 else 3
            y_start = max(0, y - window_size); y_end = min(H, y + window_size + 1)
            candidates = prev_col[y_start:y_end] + lam * np.abs(np.arange(y_start, y_end) - y)
            best_local_idx = np.argmin(candidates)
            dp[y, x] = candidates[best_local_idx] + current_probs[y]
            par[y, x] = y_start + best_local_idx
    path = np.zeros(W, int); path[-1] = np.argmin(dp[:, -1])
    for x in range(W - 2, -1, -1): path[x] = par[path[x + 1], x + 1]
    final_path = []
    for x, y in enumerate(path):
        y_start, y_end = max(0, y - 2), min(H, y + 3)
        weights = prob_map[y_start:y_end, x]
        if np.sum(weights) > 1e-5: final_path.append(H - (np.sum(weights * np.arange(y_start, y_end)) / np.sum(weights)))
        else: final_path.append(H - y)
    return np.array(final_path)

def robust_multi_point_calibration(crop, default_val=25.0):
    if crop is None or crop.size == 0: return default_val
    gray = cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8)); gray = clahe.apply(gray)
    edges_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    edges_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    peaks_x_indices, _ = find_peaks(np.sum(np.abs(edges_x), axis=0), distance=10, prominence=50)
    peaks_y_indices, _ = find_peaks(np.sum(np.abs(edges_y), axis=1), distance=10, prominence=50)
    grid_sizes = []
    if len(peaks_x_indices) > 3: grid_sizes.extend(np.diff(peaks_x_indices)[(np.diff(peaks_x_indices)>10) & (np.diff(peaks_x_indices)<100)])
    if len(peaks_y_indices) > 3: grid_sizes.extend(np.diff(peaks_y_indices)[(np.diff(peaks_y_indices)>10) & (np.diff(peaks_y_indices)<100)])
    if len(grid_sizes) >= 5:
        grid_sizes = np.array(grid_sizes)
        q1 = np.percentile(grid_sizes, 25); q3 = np.percentile(grid_sizes, 75); iqr = q3 - q1
        clean_grid = grid_sizes[(grid_sizes >= q1 - 1.5*iqr) & (grid_sizes <= q3 + 1.5*iqr)]
        if len(clean_grid) > 0: return (0.6 * np.median(clean_grid)) + (0.4 * np.mean(clean_grid))
    return default_val

def advanced_perspective_correction(image):
    if image is None: return None
    h, w = image.shape[:2]
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=100, minLineLength=w//4, maxLineGap=20)
    if lines is None: return image
    horizontal_points = []
    for line in lines:
        x1, y1, x2, y2 = line[0]; angle = np.arctan2(y2 - y1, x2 - x1) * 180 / np.pi
        if abs(angle) < 10: horizontal_points.append([x1, y1]); horizontal_points.append([x2, y2])
    if len(horizontal_points) < 10: return image
    horizontal_points = np.array(horizontal_points)
    try:
        from skimage.measure import LineModelND
        model, inliers = ransac(horizontal_points, LineModelND, min_samples=2, residual_threshold=5, max_trials=100)
        vector = model.params[1]; angle_deg = np.degrees(np.arctan2(vector[1], vector[0]))
        if abs(angle_deg) > 0.5 and abs(angle_deg) < 15:
            center = (w // 2, h // 2); M = cv2.getRotationMatrix2D(center, angle_deg, 1.0)
            return cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    except: pass
    return image

def get_yolo_crops_with_fallback(image, model):
    if model:
        try:
            results = model.predict(image, verbose=False, conf=0.15, imgsz=640)
            found = {}
            for box in results[0].boxes.cpu().numpy():
                cid = int(box.cls[0]); conf = box.conf[0]
                if cid not in found or conf > found[cid]['conf']: found[cid] = {'box': box.xyxy[0], 'conf': conf}
            if len(found) == 12:
                crops = []; h, w = image.shape[:2]
                for i in range(12):
                    x1,y1,x2,y2 = map(int, found[i]['box']); pad=5
                    crops.append(image[max(0,y1-pad):min(h,y2+pad), max(0,x1-pad):min(w,x2+pad)])
                return crops
        except: pass
    h, w = image.shape[:2]; rh = h//4; cw = w//3
    return [image[r*rh:(r+1)*rh, c*cw:(c+1)*cw] for r in range(4) for c in range(3)]

def preprocess_remove_grid_rgb(image_rgb):
    if image_rgb is None: return None
    hsv = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2HSV) 
    mask = cv2.inRange(hsv, np.array([0, 50, 50]), np.array([10, 255, 255])) + \
           cv2.inRange(hsv, np.array([170, 50, 50]), np.array([180, 255, 255]))
    mask = cv2.dilate(mask, np.ones((2,2), np.uint8), iterations=1)
    img_c = image_rgb.copy(); img_c[mask > 0] = (255, 255, 255)
    return img_c

# --- 5. التنفيذ النهائي ---
if os.path.exists(TEST_CSV_PATH): test_df = pd.read_csv(TEST_CSV_PATH)
else: test_df = pd.DataFrame({'id': ['001'], 'fs': [500], 'number_of_rows': [5000]})
paths = glob.glob(f"{IMAGE_DIR}/**/*.png") + glob.glob(f"{IMAGE_DIR}/**/*.jpg")
path_map = {os.path.splitext(os.path.basename(p))[0]: p for p in paths}
ids_list, vals_list = [], []

print(f"🚀 بدء الاستخراج البلاتيني (Phase 14+15) مع الحفظ بصيغة CSV...")

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    sid = str(row['id'])
    tlen = int(row['number_of_rows']) if 'number_of_rows' in row and not pd.isna(row['number_of_rows']) else int(10*(row.get('fs',500)))
    fs = float(row['fs']) if 'fs' in row and not pd.isna(row['fs']) else 500.0
    path = path_map.get(sid)
    leads = {}
    if path:
        try:
            img = cv2.imread(path); img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = advanced_perspective_correction(img)
            global_grid = robust_multi_point_calibration(img, 25.0) 
            crops = get_yolo_crops_with_fallback(img, yolo_model)
            crops = [preprocess_remove_grid_rgb(c) for c in crops]
            for i, crop in enumerate(crops):
                lname = LEAD_NAMES[i]
                if crop.size < 100: leads[lname] = np.zeros(tlen); continue
                
                local_grid = robust_multi_point_calibration(crop, default_val=global_grid)
                
                # Phase 15: TTA
                prob, scale = predict_with_tta(crop, unet_model)
                
                raw = adaptive_viterbi(prob)
                g_sc = local_grid * scale
                sig_mv = (raw - np.median(raw)) / (g_sc * 10 if g_sc > 1 else 250)
                
                # Phase 14: Medical Polish
                if len(sig_mv)>15: sig_mv = savgol_filter(sig_mv, 11, 3)
                if len(sig_mv) > 20: sig_mv = apply_high_pass_filter(sig_mv, cutoff=0.5, fs=fs)
                
                leads[lname] = resample(sig_mv, tlen) if len(sig_mv)>0 else np.zeros(tlen)
            
            leads = smart_einthoven_fix(leads)
            
        except: leads = {l: np.zeros(tlen) for l in LEAD_NAMES}
    else: leads = {l: np.zeros(tlen) for l in LEAD_NAMES}
    for l in LEAD_NAMES:
        s = leads.get(l, np.zeros(tlen))
        if len(s)!=tlen: s = resample(s, tlen) if len(s)>0 else np.zeros(tlen)
        ids_list.extend([f"{sid}_{i}_{l}" for i in range(tlen)])
        vals_list.extend(s)

print("💾 الحفظ النهائي (CSV Format)...")
# تحويل القوائم إلى DataFrame
df_submission = pd.DataFrame({'id': ids_list, 'value': vals_list})

# الحفظ بصيغة CSV (مهم جداً: index=False)
df_submission.to_csv("submission.csv", index=False)

print("✅ تم حفظ submission.csv بنجاح! الملف جاهز للمسابقة.")

⚙️ جاري تحميل المحرك البلاتيني (Phase 14+15)...
✅ YOLO (Combat): مفعل (تم التحميل من Input).
💎 U-Net (Phase 10): مفعل (تم التحميل من Input).
🚀 بدء الاستخراج البلاتيني (Phase 14+15) مع الحفظ بصيغة CSV...


100%|██████████| 24/24 [11:45<00:00, 29.40s/it]


💾 الحفظ النهائي (CSV Format)...
✅ تم حفظ submission.csv بنجاح! الملف جاهز للمسابقة.


In [5]:
# --- خلية 23: التحقق النهائي (Updated for CSV) ---

# 1. تحديث المسارات لتناسب الصيغة الجديدة CSV
MY_SUB_PATH = "submission.csv"
# عادةً ملف العينة يكون csv أيضاً
OFFICIAL_SAMPLE_PATH = "/kaggle/input/physionet-ecg-image-digitization/sample_submission.csv" 

print("🔍 جاري التحقق من ملف CSV (Long Format)...")

try:
    # التأكد من وجود الملف أولاً
    if not os.path.exists(MY_SUB_PATH):
        raise FileNotFoundError("ملف submission.csv غير موجود! تأكد أن الخلية 22 انتهت بنجاح.")

    # القراءة بصيغة CSV
    my_sub = pd.read_csv(MY_SUB_PATH)
    print(f"✅ تم تحميل ملفك: {MY_SUB_PATH}")

    try:
        if os.path.exists(OFFICIAL_SAMPLE_PATH):
            sample_sub = pd.read_csv(OFFICIAL_SAMPLE_PATH)
            has_sample = True
        else:
            # محاولة قراءة parquet إذا لم يوجد csv كخطة بديلة للمقارنة فقط
            sample_sub = pd.read_parquet(OFFICIAL_SAMPLE_PATH.replace('.csv', '.parquet'))
            has_sample = True
    except:
        has_sample = False
        print("⚠️ لم نتمكن من قراءة ملف العينة الرسمي للمقارنة (لا يؤثر على الحل).")

    # التحقق من أسماء الأعمدة
    required_columns = {'id', 'value'}
    my_columns = set(my_sub.columns)
    
    if my_columns != required_columns:
        print(f"❌ خطأ قاتل: الأعمدة غير صحيحة!")
        print(f"الموجود لديك: {my_columns}")
        print(f"المطلوب: {required_columns}")
        raise ValueError("Column Mismatch")
    else:
        print("✅ أسماء الأعمدة صحيحة (id, value).")

    # التحقق من نوع البيانات (float)
    first_value = my_sub.iloc[0]['value']
    # في CSV أحياناً يتم قراءة الأرقام كنص، لذا نتأكد
    if not isinstance(first_value, (float, np.floating, int, np.integer)):
        print(f"⚠️ تحذير: نوع القيم قد لا يكون float مباشرة. النوع المكتشف: {type(first_value)}")
        print("جاري محاولة التحويل للتأكد...")
        try:
            float(first_value)
            print("✅ القيم قابلة للتحويل لأرقام بنجاح.")
        except:
            print("❌ القيم ليست أرقاماً!")
            raise ValueError("Value Type Error")
    else:
        print("✅ نوع البيانات (float/number) صحيح.")

    # التحقق من شكل الـ ID
    first_id = str(my_sub.iloc[0]['id'])
    print(f"🔍 عينة من الـ ID في ملفك: {first_id}")
    if "_" not in first_id:
        print("⚠️ تحذير: شكل الـ ID يبدو غريباً (لا يحتوي على underscore).")
    
    # التحقق من عدد الصفوف
    print(f"📊 إجمالي عدد الصفوف في ملفك: {len(my_sub)}")
    
    if has_sample:
        print(f"📊 إجمالي عدد الصفوف في Sample الرسمي: {len(sample_sub)}")
        if len(my_sub) == len(sample_sub):
            print("✅ تطابق تام في عدد الصفوف!")
        else:
            print("ℹ️ عدد الصفوف مختلف (وهذا طبيعي إذا كان الاختبار مخفياً Hidden Test Set).")
    
    if len(my_sub) % 12 == 0:
            print("✅ عدد الصفوف يقبل القسمة على 12 (منطقي لـ 12 leads).")
    else:
            print("⚠️ تحذير: عدد الصفوف لا يقبل القسمة على 12!")

    print("\n🎉 النتيجة النهائية: ملف submission.csv سليم 100% وجاهز للرفع.")
    print("توكل على الله واضغط Submit!")

except Exception as e:
    print(f"\n❌ حدث خطأ أثناء التحقق: {e}")

🔍 جاري التحقق من ملف CSV (Long Format)...
✅ تم تحميل ملفك: submission.csv
✅ أسماء الأعمدة صحيحة (id, value).
✅ نوع البيانات (float/number) صحيح.
🔍 عينة من الـ ID في ملفك: 1053922973_0_I
📊 إجمالي عدد الصفوف في ملفك: 900000
📊 إجمالي عدد الصفوف في Sample الرسمي: 75000
ℹ️ عدد الصفوف مختلف (وهذا طبيعي إذا كان الاختبار مخفياً Hidden Test Set).
✅ عدد الصفوف يقبل القسمة على 12 (منطقي لـ 12 leads).

🎉 النتيجة النهائية: ملف submission.csv سليم 100% وجاهز للرفع.
توكل على الله واضغط Submit!
